# Training Results Viewer (macro evaluation)

This notebook allows you to load and visualize the results of finetuning LaBraM on the EEG dataset. It generates figures based on averaged results across datasets, allowing to compare the general performance of one or more models.

## Setup

In [ ]:
import numpy as np
from IPython.display import display
from pathlib import Path
import json
import core.settings as settings
import core.utils as core_utils
import matplotlib.pyplot as plt
import os
import core.io as io
from ipywidgets import widgets
from libs.labram import utils
from libs.labram import engine_for_finetuning as engine
from torch import utils as torch_utils
import torch

from core.architectures import EEGAutoencoder, EEGSpectralDecoder
from core.datasets import KFoldSplitter

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir('..')

def save_button(fig, save_dir: Path, default_name="untitled"):
    name_input = widgets.Text(value=default_name, description='Filename:', placeholder='untitled')

    save_button_comp = widgets.Button(description="Save Figure")
    output_comp = widgets.Output()
    def on_save_clicked(b):
        with output_comp:
            output_comp.clear_output()
            filename = name_input.value.strip() or default_name
            if not filename.endswith(".png"):
                filename += ".png"
            save_dir.mkdir(parents=True, exist_ok=True)

            try:
                save_path = save_dir / filename
                fig.savefig(save_path, dpi=300, bbox_inches='tight')
                print(f"Figure saved to: {save_path}")
            except Exception as e:
                print(f"Error saving figure: {e}")
    save_button_comp.on_click(on_save_clicked)
    display(widgets.VBox([widgets.HBox([name_input]), save_button_comp, output_comp]))

def show_delta(ax, epoch, wave1, wave2):
    val1 = wave1[epoch]
    val2 = wave2[epoch]
    delta = np.absolute(val2 - val1)
    midpoint = (val1 + val2) / 2
    ax.plot(epoch, val1, 'ko', markersize=4)
    ax.plot(epoch, val2, 'ko', markersize=4)

    ax.vlines(x=epoch, ymin=min(val1, val2), ymax=max(val1, val2), color='black', linestyle='--')
    ax.annotate(
        f"$\Delta$: {delta:.4f}",
        xy=(epoch, midpoint),
        xytext=(10, 0),
        textcoords='offset points',
        va='center',
        bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='black', alpha=0.8),
        color='black',
        fontsize=10
    )



## Plotting training and validation losses across epochs

In [ ]:
model_name = "fixed_wavelet_huber"

model_data = io.load_model_log_data(settings.FINETUNE_OUTPUT_DIR, model_name=model_name)

avg_metrics = core_utils.avg_metrics_over_folds(model_data = model_data, metrics=["train_r2", "val_r2"])

avg_train = avg_metrics["train_r2"]
avg_val = avg_metrics["val_r2"]

# plot training and validation losses across epochs
fig = plt.figure(figsize=(10, 6))
plt.plot(avg_train, label="Train R2")
plt.plot(avg_val, label="Validation R2")
plt.xlabel("Epoch")
plt.ylabel("R2 Score")
plt.title(f"Training and Validation R2 Scores for {model_name}")
plt.legend()
plt.grid()
save_button(fig, save_dir=settings.PLOT_MACRO/"train_val", default_name=model_name)
max_val_epoch = np.argmax(avg_val)
show_delta(fig.gca(), max_val_epoch, avg_train, avg_val)
plt.show()

## Plotting val performance of multiple models across epochs

In [ ]:
model_names = ["fixed_wavelet_huber", "huber_0929", "unbalanced_huber","balanced_huber"]
#model_names = ["wavelet_5e-3_pre", "wavelet_5e-3_scratch"]
#model_names= ["wavelet_1e-3_scratch", "wavelet_5e-3_scratch", "wavelet_5e-4_scratch"]
#model_names = ["wavelet_1e-3_pre", "wavelet_1e-3_scratch", "wavelet_5e-3_pre", "wavelet_5e-3_scratch", "wavelet_5e-4_pre", "wavelet_5e-4_scratch"]
#compare_diff = [("wavelet_5e-3_pre", "wavelet_5e-4_pre")]
compare_diff = []
to = 80

avg_vals = {}
for model_name in model_names:
    model_data = io.load_model_log_data(settings.FINETUNE_OUTPUT_DIR, model_name=model_name)

    avg_metrics = core_utils.avg_metrics_over_folds(model_data = model_data, metrics=["val_r2"])
    avg_vals[model_name] = avg_metrics["val_r2"]

#plot validation losses across epochs on one line graph for all models
fig = plt.figure(figsize=(10, 6))
for model_name in model_names:
    if "pre" in model_name:
        label = f"{model_name} (pretrained)"
        linestyle = "--"
    else:
        label = model_name
        linestyle = "-"
    plt.plot(avg_vals[model_name][:to], label=label, linestyle=linestyle)

for model_a, model_b in compare_diff:
    val_a = avg_vals[model_a]
    val_b = avg_vals[model_b]
    show_delta(fig.gca(), min(len(val_a), len(val_b))-1, val_a, val_b)

plt.xlabel("Epoch")
plt.ylabel("Validation R2 Score")
plt.title("Validation R2 Scores Across Epochs for Multiple Models")
plt.legend()
plt.grid()
save_button(fig, save_dir=settings.PLOT_MACRO/"val_multiple")
plt.show()


## Comparing models' performance on test set

In [ ]:
%%script false --no-raise-error
# compared models are expected to have the same amount of folds
#model_names = ["wavelet_1e-3_pre", "wavelet_1e-3_scratch", "wavelet_5e-3_pre", "wavelet_5e-3_srcratcrh", "wavelet_5e-4_pre", "wavelet_5e-4_scratch"]
model_names = ["llrd_norm_5e-3", "llrd_5e-3"]
log_norm = [(True, True), (True, False)]
splitter = KFoldSplitter(input_data_dir=settings.FINETUNE_EPOCHS_DIR, seed=settings.SEED, k_folds=6, no_press_threshold=settings.NO_PRESS_FROM_SUBJECT)
folds = splitter.split_by_subject()

ch_names = settings.CHORDER_STANDARD
# cut off "-REF" and "EEG " from channel names for compatibility with get_input_chans function
ch_names = [name.split(' ')[-1].split('-')[0] for name in ch_names]

model_r2s = []
for idx, model_name in enumerate(model_names):
    fold_means = []
    log=log_norm[idx][0]
    norm=log_norm[idx][1]
    for i, fold in enumerate(folds):
        with open(settings.FINETUNE_OUTPUT_DIR / model_name / "metadata.json", 'r') as f:
            metadata = json.load(f)
        target_mode =  "wavelet" if "n_freqs" in metadata["model_params"] else "autoencoder"

        _, _, test_dataset = fold.initialize_datasets(target_mode=target_mode,
                                                      window_size=settings.TARGET_WINDOW_SIZE, wavelet_dir=settings.FINETUNE_WAVELET_DIR, superlet_dir=settings.FINETUNE_SUPERLET_DIR, log=log, normalize=norm)
        test_sampler = torch_utils.data.SequentialSampler(test_dataset)
        test_loader = torch_utils.data.DataLoader(test_dataset, sampler = test_sampler, batch_size=256)
        model = None
        model_params = None
        if target_mode == "autoencoder":
            model_params = settings.AUTOENCODER_PARAMS
            model = EEGAutoencoder(**model_params)
        elif target_mode in ["wavelet", "superlet"]:
            model_params = settings.SPECTRAL_DECODER_PARAMS
            model = EEGSpectralDecoder(**model_params)
        else:
            raise ValueError(f"Invalid target mode: {target_mode}")

        # loading checkpoint into model
        checkpoint_path = settings.FINETUNE_OUTPUT_DIR / model_name / str(i+1) / "checkpoint-best_val.pth"
        print("using checkpoint:", checkpoint_path)
        if not checkpoint_path.exists():
            continue
        checkpoint = torch.load(checkpoint_path, map_location="cpu")
        model.load_state_dict(checkpoint["model"])

        # set model to eval mode and move to device
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        model.to(device)
        model.eval()

        test_stats = engine.evaluate(test_loader, model, device, header="Test:", ch_names=ch_names, metrics=["mse", "mae", "r2"], is_binary = False)
        test_r2 = test_stats["r2"]
        fold_means.append(test_r2)
    overall_mean = np.mean(fold_means)
    print(f"Model: {model_name}, Mean Test R2 across folds: {overall_mean:.4f}")
    model_r2s.append(overall_mean)

#plot bar graph comparing models' mean test R2 scores
fig = plt.figure(figsize=(10, 6))
ax1 = fig.add_subplot(211)
ax1.bar(model_names, model_r2s, color='skyblue')
ax1.set_xlabel("Model")
ax1.set_ylabel("Mean Test R2 Score")
ax1.set_title("Comparison of Mean Test R2 Scores Across Models")
ax1.grid(axis='y')

ax2 = fig.add_subplot(212)
ax2.axis('off')

table = ax2.table(cellText=model_r2s, rowLabels=model_names, loc='center')

save_button(fig, save_dir=settings.PLOT_MACRO/"test_comparison")
plt.show()


        # sample_r2s = []
        # for test_item in test_dataset:
        #     trial_data = test_item[0]
        #     test_input = core_utils.format_model_input(trial_data, device=device)
        #
        #     target_data = test_item[1]
        #
        #     reconstruction, epoch_idx = core_utils.reconstruct_model_output(checkpoint_path=checkpoint_path, model=model, input=test_input, input_chans=input_chans, window_size=settings.WINDOW_SIZE, **model_params)
        #     channel_r2s = []
        #     for ch_idx in range(reconstruction.shape[0]):
        #         channel_true = target_data[ch_idx].cpu().numpy()
        #         channel_recon = reconstruction[ch_idx].cpu().numpy()
        #
        #         r2 = core_utils.calculate_r2(channel_true, channel_recon)
        #         channel_r2s.append(r2)
        #
        #     sample_mean = np.mean(channel_r2s)
        #     sample_r2s.append(sample_mean)
        # fold_mean = np.mean(sample_r2s)
        # print(f"Model: {model_name}, Fold: {i+1}, Test R2: {fold_mean:.4f}")



